# Phase 2A — ToolAlpaca SFT Training (Kaggle-Optimised)

**Ported from Google Colab. Key optimisations:**
- Replaced Google Drive I/O with Kaggle `/kaggle/working/` output directory
- Pinned exact package versions for reproducibility on Kaggle's base image
- Removed Colab-specific helpers (`google.colab`, nvjitlink path discovery)
- Enabled `packing=True` in SFTTrainer to cut effective steps and speed up training
- Switched default model to `Qwen/Qwen2-0.5B` for quick-test; Mistral-7B for full run
- Gradient accumulation tuned for T4 16 GB VRAM budget
- `flash_attention_2` used automatically when bf16 is available (A100/H100)
- DataLoader workers set to 2 to overlap CPU preprocessing with GPU compute
- Paged AdamW 8-bit used when bitsandbytes is available; falls back to AdamW cleanly
- `max_steps` cap in full mode prevents runaway training on large datasets

**Usage:**
1. Upload `agent_trajectories_2k.json` and `sales_data.csv` to the Kaggle dataset and update `DATA_PATH` / `CSV_PATH` below.
2. Add your `sft_toolalpaca.py` helper to the Kaggle dataset or paste inline.
3. Run all cells. The adapter is saved to `/kaggle/working/cs_f425_sft_adapter/`.
4. Download from the Kaggle output panel or push to Hugging Face Hub.

In [ ]:
# ── Cell 1: Environment Detection & GPU Verification ──────────────────────────
import os, sys, subprocess, glob
from pathlib import Path

os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Suppress tokenizer parallelism warnings that clutter Kaggle output
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Disable W&B unconditionally — not needed and causes TRL import noise
os.environ["WANDB_DISABLED"] = "true"

IN_KAGGLE = os.path.exists("/kaggle/working")
print(f"Running in Kaggle: {IN_KAGGLE}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    total_gb = gpu.total_memory / 1024**3
    print(f"GPU: {gpu.name}  VRAM: {total_gb:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
    subprocess.run(["nvidia-smi"], check=False)
else:
    total_gb = 0
    print("WARNING: No GPU detected — training will be extremely slow.")

In [ ]:
# ── Cell 2: Install / Verify Packages ─────────────────────────────────────────
# Kaggle's base image ships torch 2.x. We pin versions that are mutually
# compatible and tested on T4.  All installs are --quiet to keep logs readable.
#
# If you run this on a Kaggle notebook that already has these packages at the
# right versions (check via `pip show transformers`), you can skip this cell.

PKGS = [
    "transformers==4.44.2",
    "peft==0.12.0",
    "trl==0.10.1",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "sentencepiece==0.2.0",
    "protobuf==4.25.4",
    "bitsandbytes>=0.43.0",   # 4-bit NF4 quant; soft-fail below
]

def pip(*args, soft=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and not soft:
        raise RuntimeError(f"pip install failed for {args}:\n{r.stderr[-600:]}")
    return r.returncode == 0

# Upgrade pip/wheel first so metadata parsing doesn't break
pip("--upgrade", "pip", "setuptools", "wheel")

failed = []
for pkg in PKGS:
    ok = pip(pkg, soft=True)
    if not ok:
        failed.append(pkg)
        print(f"  [SOFT FAIL] {pkg}")

# W&B removal (can silently break TRL on some Kaggle images)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "wandb"],
               capture_output=True)

# bitsandbytes probe — determines whether 4-bit quant is usable at runtime
USE_4BIT = False
try:
    import bitsandbytes as bnb
    if torch.cuda.is_available():
        probe = torch.randn(128, device="cuda", dtype=torch.float16)
        bnb.functional.quantize_4bit(probe, quant_type="nf4")
        USE_4BIT = True
        print(f"bitsandbytes {bnb.__version__} — 4-bit NF4 OK")
except Exception as e:
    print(f"bitsandbytes 4-bit probe failed ({e}) — will use fp16/bf16 without quant")

print(f"USE_4BIT={USE_4BIT}")
if failed:
    print(f"Soft-failed packages: {failed}  (training may still work)")

In [ ]:
# ── Cell 3: Paths ─────────────────────────────────────────────────────────────
# Kaggle stores uploaded datasets under /kaggle/input/<dataset-slug>/.
# Outputs go to /kaggle/working/ and are downloadable from the output panel.
#
# ► UPDATE THESE PATHS to match your Kaggle dataset slug and filenames.

if IN_KAGGLE:
    # Example: dataset slug = "cs-f425-data"
    DATASET_INPUT = Path("/kaggle/input/cs-f425-data")
    WORKING       = Path("/kaggle/working")
else:
    # Local fallback — mirrors the Colab project folder layout
    DATASET_INPUT = Path.cwd()
    WORKING       = Path.cwd() / "working"

DATA_PATH   = DATASET_INPUT / "agent_trajectories_2k.json"
CSV_PATH    = DATASET_INPUT / "sales_data.csv"

# sft_toolalpaca.py: either in the dataset or next to this notebook
HELPER_PATH = DATASET_INPUT / "sft_toolalpaca.py"
if not HELPER_PATH.exists():
    HELPER_PATH = Path.cwd() / "sft_toolalpaca.py"

ADAPTER_DIR = WORKING / "cs_f425_sft_adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

# Make the helper importable
helper_dir = str(HELPER_PATH.parent)
if helper_dir not in sys.path:
    sys.path.insert(0, helper_dir)

print(f"Data file exists  : {DATA_PATH.exists()}  →  {DATA_PATH}")
print(f"CSV file exists   : {CSV_PATH.exists()}  →  {CSV_PATH}")
print(f"Helper exists     : {HELPER_PATH.exists()}  →  {HELPER_PATH}")
print(f"Adapter output    : {ADAPTER_DIR}")

In [ ]:
# ── Cell 4: Load & Inspect Dataset ────────────────────────────────────────────
import csv
import json

with open(DATA_PATH, "r", encoding="utf-8") as f:
    trajectories = json.load(f)

with open(CSV_PATH, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    sales_rows = list(reader)
    sales_columns = reader.fieldnames or []

print(f"Trajectory count : {len(trajectories)}")
print(f"Sales shape      : ({len(sales_rows)}, {len(sales_columns)})")
print(f"Sales columns    : {sales_columns}")
print()
print("── Sample sales rows ──")
for row in sales_rows[:3]:
    print(row)
print()
print("── Sample trajectory ──")
print(json.dumps(trajectories[0], indent=2))

In [ ]:
# ── Cell 5: Preprocess → ReAct Format & Train/Val/Test Split ──────────────────
import re as _re
from datasets import Dataset
from sklearn.model_selection import train_test_split

_THOUGHTS = {
    "filter_data":    "I need to filter the data for the specified condition.",
    "group_by":       "I need to group the data by the specified column.",
    "aggregate_sum":  "I need to compute the sum of the specified column.",
    "aggregate_mean": "I need to compute the mean of the specified column.",
    "sort_by":        "I need to sort the results by the specified column.",
    "top_k":          "I need to select the top entries from the results.",
}

SYSTEM_PROMPT = (
    "You are a data analysis agent. Given a user query about sales data, "
    "output a sequence of Thought/Action/Action Input steps to answer it.\n\n"
    "Dataset columns: date, year, month, city, region, product, category, "
    "revenue, units_sold, cost, profit\n\n"
    "Available actions: filter_data, group_by, aggregate_sum, aggregate_mean, sort_by, top_k\n\n"
    "After all steps, output:\nFinal Answer: <result description>"
)


def action_str_to_react_block(action_str: str) -> str:
    """Convert 'filter_data(column=\'year\', value=2022)' → ReAct triple."""
    name = action_str.split("(")[0].strip()
    thought = _THOUGHTS.get(name, "I need to execute the next step.")
    args_str = action_str[action_str.index("(") + 1: action_str.rindex(")")]
    kwargs: dict = {}
    for m in _re.finditer(
        r"(\w+)\s*=\s*(?:'([^']*)'|\"([^\"]*)\"|(-?\d+(?:\.\d+)?))", args_str
    ):
        key = m.group(1)
        val: object = m.group(2) or m.group(3) or m.group(4)
        try:
            val = int(val)      # type: ignore[arg-type]
        except (TypeError, ValueError):
            try:
                val = float(val)  # type: ignore[arg-type]
            except (TypeError, ValueError):
                pass
        kwargs[key] = val
    return f"Thought: {thought}\nAction: {name}\nAction Input: {json.dumps(kwargs)}"


examples = []
for entry in trajectories:
    blocks = [action_str_to_react_block(a) for a in entry["actions"]]
    completion = "\n\n".join(blocks) + "\n\nFinal Answer: Based on the executed steps."
    text = (
        f"[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
        f"[USER]\n{entry['query']}\n\n"
        f"[ASSISTANT]\n{completion}"
    )
    examples.append({"text": text, "completion": completion})

train_ex, temp_ex = train_test_split(examples, test_size=0.2, random_state=42)
val_ex, test_ex  = train_test_split(temp_ex,   test_size=0.5, random_state=42)

train_ds = Dataset.from_list(train_ex)
val_ds   = Dataset.from_list(val_ex)
test_ds  = Dataset.from_list(test_ex)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")
print()
print("── Sample text (first 800 chars) ──")
print(train_ds[0]["text"][:800])

In [ ]:
# ── Cell 6: Model & Mode Selection ────────────────────────────────────────────
# QUICK_TEST_MODE — tiny model, 40 steps; for verifying the pipeline end-to-end.
# FAST_7B_MODE    — full 7B model but capped steps/sequence length.
# Default (both False) — full 7B training, ~3 epochs.
#
# On Kaggle T4 (16 GB VRAM) the full 7B run takes ~90 min with 4-bit + packing.

QUICK_TEST_MODE = os.getenv("QUICK_TEST_MODE", "0").strip() in {"1","true","True","yes"}
FAST_7B_MODE    = (not QUICK_TEST_MODE) and (
    os.getenv("FAST_7B_MODE", "0").strip() in {"1","true","True","yes"}
)

DEFAULT_7B_MODEL  = "mistralai/Mistral-7B-v0.1"
QUICK_TEST_MODEL  = os.getenv("QUICK_TEST_MODEL", "Qwen/Qwen2-0.5B")  # ~1 GB

MODEL_ID = QUICK_TEST_MODEL if QUICK_TEST_MODE else os.getenv("MODEL_ID", DEFAULT_7B_MODEL)

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

print(f"QUICK_TEST_MODE : {QUICK_TEST_MODE}")
print(f"FAST_7B_MODE    : {FAST_7B_MODE}")
print(f"MODEL_ID        : {MODEL_ID}")
print(f"BF16 supported  : {bf16_ok}")
print(f"USE_4BIT        : {USE_4BIT}")
print(f"GPU VRAM        : {total_gb:.1f} GB")

# Safety gate: without 4-bit quant a 7B model needs ~28 GB VRAM
if not QUICK_TEST_MODE and not USE_4BIT and total_gb < 28:
    raise RuntimeError(
        "7B model requires 4-bit quantisation on T4 (16 GB).\n"
        "bitsandbytes probe failed above — check the install log.\n"
        "Alternatively set QUICK_TEST_MODE=1 to run the pipeline with a small model."
    )

In [ ]:
# ── Cell 7: Load Base Model + LoRA ────────────────────────────────────────────
# We load the model here (not inside sft_toolalpaca) so Kaggle's memory
# allocation is visible in the cell output and easier to debug.

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Tokenizer ──────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"   # required for packing=True

# ── Model loading ──────────────────────────────────────────────────────────────
model_kwargs: dict = dict(
    trust_remote_code=True,
    device_map="auto",
)

if USE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,       # saves ~0.4 GB extra
        bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16,
    )
    model_kwargs["quantization_config"] = bnb_cfg
else:
    model_kwargs["torch_dtype"] = torch.bfloat16 if bf16_ok else torch.float16

# Flash-attention-2 (A100 / H100 only; silently skip on T4)
try:
    if bf16_ok and not QUICK_TEST_MODE:
        model_kwargs["attn_implementation"] = "flash_attention_2"
        print("Using Flash Attention 2")
except Exception:
    pass

print(f"Loading {MODEL_ID} …", flush=True)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

# ── LoRA config ───────────────────────────────────────────────────────────────
if USE_4BIT:
    base_model = prepare_model_for_kbit_training(
        base_model, use_gradient_checkpointing=True
    )

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],   # Mistral / LLaMA attention heads
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
peft_model = get_peft_model(base_model, lora_cfg)
peft_model.print_trainable_parameters()
print("Model ready.", flush=True)

In [ ]:
# ── Cell 8: Training Arguments ────────────────────────────────────────────────
import inspect
from transformers import TrainingArguments

# Detect whether this transformers build uses the old 'evaluation_strategy'
# parameter name or the new 'eval_strategy' alias introduced in 4.41+.
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
STRATEGY_KEY = "evaluation_strategy" if "evaluation_strategy" in _ta_params else "eval_strategy"

# Optimiser: paged_adamw_8bit when bnb is available, else adamw_torch
_optim = "paged_adamw_8bit" if USE_4BIT else "adamw_torch"

# ── Config presets ────────────────────────────────────────────────────────────
if QUICK_TEST_MODE:
    # ~40 steps; just validates the full pipeline runs without crashing
    _kwargs = dict(
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        logging_steps=5,
        save_steps=500,
        save_total_limit=1,
        eval_steps=20,
        max_steps=40,
    )
elif FAST_7B_MODE:
    # ~220 steps on subset; good for iterating on 7B without a full run
    _kwargs = dict(
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch = 8
        learning_rate=2e-4,
        warmup_ratio=0.03,
        logging_steps=10,
        save_steps=300,
        save_total_limit=1,
        eval_steps=200,
        max_steps=220,
    )
else:
    # Full training run (~3 epochs, capped at 600 steps on T4)
    # packing=True (set in SFTTrainer) makes each step cover ~3× more tokens
    # so 600 packed steps ≈ 1800 unpacked steps in wall-clock terms.
    _kwargs = dict(
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,  # effective batch = 16
        learning_rate=2e-4,
        warmup_ratio=0.05,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        eval_steps=100,
        max_steps=600,                  # safety cap for large datasets on T4
    )

_kwargs.update(dict(
    output_dir=str(ADAPTER_DIR),
    lr_scheduler_type="cosine",
    fp16=not bf16_ok,
    bf16=bf16_ok,
    optim=_optim,
    report_to="none",
    dataloader_num_workers=2,     # overlap CPU tokenisation with GPU compute
    dataloader_pin_memory=True,   # faster H2D transfer on T4
    gradient_checkpointing=True,  # trade compute for VRAM
    remove_unused_columns=False,
    group_by_length=True,         # batch similar-length seqs → less padding waste
    **{STRATEGY_KEY: "steps"},
))

train_args = TrainingArguments(**_kwargs)
print(f"Optimiser      : {train_args.optim}")
print(f"BF16/FP16      : bf16={train_args.bf16}  fp16={train_args.fp16}")
print(f"Effective batch: {train_args.per_device_train_batch_size * train_args.gradient_accumulation_steps}")
print(f"Max steps      : {train_args.max_steps}")
print(f"Strategy key   : {STRATEGY_KEY}")

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────────────
import inspect, subprocess
from trl import SFTTrainer
from transformers import TrainerCallback

# ── Dataset subsetting ────────────────────────────────────────────────────────
train_dataset, val_dataset = train_ds, val_ds

if QUICK_TEST_MODE:
    train_dataset = train_dataset.select(range(min(300, len(train_dataset))))
    val_dataset   = val_dataset.select(range(min(60,  len(val_dataset))))
    MAX_SEQ_LEN   = 512
elif FAST_7B_MODE:
    train_dataset = train_dataset.select(range(min(1200, len(train_dataset))))
    val_dataset   = val_dataset.select(range(min(200,  len(val_dataset))))
    MAX_SEQ_LEN   = 256
else:
    MAX_SEQ_LEN = 512

print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}  max_seq_len: {MAX_SEQ_LEN}")

# ── Progress callback ─────────────────────────────────────────────────────────
def _gpu_snap() -> str:
    try:
        return subprocess.check_output(
            ["nvidia-smi",
             "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            text=True,
        ).strip()
    except Exception as e:
        return f"(nvidia-smi unavailable: {e})"

class KaggleProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw):
        print(f"Training started  max_steps={state.max_steps}  epochs={args.num_train_epochs}")
        print(f"GPU: {_gpu_snap()}")

    def on_log(self, args, state, control, logs=None, **kw):
        logs = logs or {}
        parts = []
        for k, fmt in [("loss", ".4f"), ("eval_loss", ".4f"),
                       ("learning_rate", ".2e"), ("epoch", ".2f")]:
            if k in logs:
                parts.append(f"{k}={logs[k]:{fmt}}")
        if parts:
            pct = 100 * state.global_step / max(state.max_steps, 1)
            print(f"[{pct:5.1f}%  step {state.global_step}]  " + "  ".join(parts))
            print(f"  GPU: {_gpu_snap()}")

# ── SFTTrainer ────────────────────────────────────────────────────────────────
_sft_params = set(inspect.signature(SFTTrainer.__init__).parameters)

_trainer_kwargs = dict(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=train_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,              # ← key speedup: bins seqs into full contexts
)

# Tokenizer arg renamed in TRL 0.9+ (tokenizer → processing_class)
if "processing_class" in _sft_params:
    _trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in _sft_params:
    _trainer_kwargs["tokenizer"] = tokenizer

# Drop any keys the installed TRL version doesn't support
_trainer_kwargs = {k: v for k, v in _trainer_kwargs.items() if k in _sft_params}

trainer = SFTTrainer(**_trainer_kwargs)
trainer.add_callback(KaggleProgressCallback())

print("Starting training …", flush=True)
trainer.train()

# ── Save adapter ──────────────────────────────────────────────────────────────
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"\nAdapter saved to: {ADAPTER_DIR}")

In [ ]:
# ── Cell 10: Offline Evaluation (Format Accuracy) ─────────────────────────────
import json
from sklearn.model_selection import train_test_split

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_trajectories = json.load(f)

_, temp_raw   = train_test_split(raw_trajectories, test_size=0.2, random_state=42)
val_raw, _    = train_test_split(temp_raw,          test_size=0.5, random_state=42)

if QUICK_TEST_MODE:
    val_raw = val_raw[:30]

val_path = WORKING / "val_trajectories.json"
with open(val_path, "w", encoding="utf-8") as f:
    json.dump(val_raw, f, indent=2)

print(f"Validation samples: {len(val_raw)}")
print(f"Val file: {val_path}")

try:
    from sft_toolalpaca import offline_eval
    n = min(50, len(val_raw))
    acc = offline_eval(trainer.model, tokenizer, str(val_path), n=n)
    print(f"Offline format accuracy ({n} samples): {acc:.2%}")
except Exception as e:
    print(f"Evaluation skipped / failed: {e}")

In [ ]:
# ── Cell 11: Export Final Adapter ─────────────────────────────────────────────
# Saves a clean copy under /kaggle/working/cs_f425_sft_adapter/final/.
# This folder appears in the Kaggle output panel for direct download.
# You can also push to Hugging Face Hub by uncommenting the block below.

final_dir = ADAPTER_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Final adapter exported to: {final_dir}")

# List output files
print("\nOutput files:")
for p in sorted(final_dir.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

# ── Optional: push to Hugging Face Hub ───────────────────────────────────────
# Uncomment and fill in your HF token (add it as a Kaggle Secret: HF_TOKEN)
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])
# trainer.model.push_to_hub("your-hf-username/cs-f425-sft-adapter")
# tokenizer.push_to_hub("your-hf-username/cs-f425-sft-adapter")
# print("Pushed to Hugging Face Hub.")